## P35 gradient field
Compute the in-plane gradient field (dL/dα, dL/dβ) over one of the P33_35a hi-res landscapes, one forward+backward per grid point (same approach as the *Gradient Fields* section of landscape_hacking_8b). Directions, image, and loss are set up exactly as in P33_35a so the field lines up with the existing `hires_option1` render.

Target: `plain74_first4  img 20907  dirseed 37  grid 512  extent 1.5`

Saves to `p35_gradient_field/<cfg>/`: an `.npz` (alphas, betas, Z, Ga, Gb, Gnorm), a `.json` of metadata, and a quiver preview png.

In [ ]:
import json, gc, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from torchvision.models.resnet import ResNet, BasicBlock
import torchvision.datasets as dsets, torchvision.transforms as T
from PIL import Image
from IPython.display import display

plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['font.family'] = 'serif'

HACKIN = Path("/home/stephen/Stephencwelch Dropbox/welch_labs/resnet/hackin")
RUNS   = HACKIN / "aug_17_run"
DATA   = Path("/home/stephen/imagenet")
OUT    = HACKIN / "p35_gradient_field"
P33_HIRES = HACKIN / "P33_landscapes_v7" / "hires_option1"   # existing 512 render, used for a sanity check only
device = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_DEPTHS = {"plain8": 8, "plain14": 14, "plain20": 20, "plain26": 26,
                "plain34": 34, "plain56": 56, "plain74": 74, "resnet74": 74}
print(torch.__version__, device)

### Model, data, directions (verbatim from P33_35a)

In [ ]:
class PlainBasicBlock(BasicBlock):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.downsample = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        return out

LAYER_CFG = {8: [1,1,1,1], 14: [2,1,1,2], 20: [2,2,3,2], 26: [3,3,3,3],
             34: [4,4,4,4], 56: [3,4,17,3], 74: [3,4,26,3]}

def make_net(depth_target, use_skip, num_classes=1000):
    block = BasicBlock if use_skip else PlainBasicBlock
    model = ResNet(block, LAYER_CFG[depth_target], num_classes=num_classes)
    if depth_target == 8:
        model.layer4 = nn.Identity()
        model.fc = nn.Linear(256, num_classes)
    return model

def load_model(name, step=None):
    d = RUNS / name
    path = d / "final.pt" if step is None else d / f"ckpt_step{step:06d}.pt"
    model = make_net(MODEL_DEPTHS[name], use_skip=name.startswith("resnet"))
    model.load_state_dict(torch.load(path, map_location=device))
    return model.to(device).eval()

def weighted_layers(model):
    '''(name, module) for convs + fc in forward order, excluding 1x1 shortcuts.'''
    return [(n, m) for n, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear)) and "downsample" not in n]

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
val_ds = dsets.ImageFolder(DATA / "ILSVRC/Data/CLS-LOC/val",
    T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)]))

def load_image(idx):
    x, y = val_ds[idx]
    return x.unsqueeze(0).to(device), torch.tensor([y], device=device)

crit = nn.CrossEntropyLoss()

LANDSCAPES = {
    "plain8_first3":   ("plain8",   [0, 1, 2]),
    "plain8_last3":    ("plain8",   [-3, -2, -1]),
    "plain26_first4":  ("plain26",  [0, 1, 2, 3]),
    "plain74_first4":  ("plain74",  [0, 1, 2, 3]),
    "plain74_last4":   ("plain74",  [-4, -3, -2, -1]),
    "resnet74_first4": ("resnet74", [0, 1, 2, 3]),
    "resnet74_last4":  ("resnet74", [-4, -3, -2, -1]),
    "plain74_first8":  ("plain74",  [0, 1, 2, 3, 4, 5, 6, 7]),
    "resnet74_first8": ("resnet74", [0, 1, 2, 3, 4, 5, 6, 7]),
    "plain74_last8":   ("plain74",  [-8, -7, -6, -5, -4, -3, -2, -1]),
    "resnet74_last8":  ("resnet74", [-8, -7, -6, -5, -4, -3, -2, -1]),
}

In [ ]:
def filter_normalize_(d, w):
    """In-place: rescale each filter of d (row for Linear) to match w's filter norm."""
    d2, w2 = d.flatten(1), w.flatten(1)
    d2.mul_(w2.norm(dim=1, keepdim=True) / (d2.norm(dim=1, keepdim=True) + 1e-10))
    return d

def make_directions(model, layer_idxs, seed):
    """Two filter-normalized random directions over ONLY the selected layers (Li et al. 2018)."""
    g = torch.Generator().manual_seed(seed)
    layers = weighted_layers(model)
    dirs = []
    for _ in range(2):
        d = {}
        for i in layer_idxs:
            name, m = layers[i]
            r = torch.randn(m.weight.shape, generator=g).to(device)
            d[name] = filter_normalize_(r, m.weight.data)
        dirs.append(d)
    return dirs  # [delta, eta]

### Target

In [ ]:
TARGET = dict(cfg="plain74_first4", img_idx=20907, dir_seed=37, grid=512, extent=1.5, loss_cap=256)

CHECKPOINT_EVERY = 16     # rows; partial results go to a *_partial.npz so a crash/interrupt is resumable
SKIP_EXISTING    = True   # if the final .npz already exists, load it instead of recomputing

def field_paths(t, root=OUT):
    d = root / t["cfg"]; d.mkdir(parents=True, exist_ok=True)
    stem = f'{t["cfg"]}_img{t["img_idx"]:06d}_dir{t["dir_seed"]:04d}_grid{t["grid"]}_ext{t["extent"]:g}_gradfield'
    return {"npz": d / f"{stem}.npz", "partial": d / f"{stem}_partial.npz",
            "meta": d / f"{stem}.json", "preview": d / f"{stem}_preview.png"}

paths = field_paths(TARGET)
print(paths["npz"])

### Gradient field

`Z[i, j]`, `Ga[i, j]`, `Gb[i, j]` are the loss, dL/dα, and dL/dβ at `alpha = lin[j]`, `beta = lin[i]` — same layout as the P33 landscapes, so `imshow(rot90(Z.T))` gives the same orientation as the existing textures. `Gnorm` is the norm of the full weight gradient over the perturbed layers (all directions, not just the plane), handy for coloring / seeing how much of the gradient lives in-plane. Gradients are raw (ascent direction); negate for descent arrows.

No autocast, matching P33_35a's `loss_surface`, so Z should reproduce the existing 512 render exactly.

In [ ]:
def grad_surface(model, delta, eta, x, y, grid_n, extent, partial_path=None, checkpoint_every=16):
    lin = np.linspace(-extent, extent, grid_n)
    layers = dict(weighted_layers(model))
    names = list(delta)
    orig = {n: layers[n].weight.data.clone() for n in names}

    Z = np.full((grid_n, grid_n), np.nan); Ga = Z.copy(); Gb = Z.copy(); Gn = Z.copy()
    start_row = 0
    if partial_path is not None and partial_path.exists():
        p = np.load(partial_path)
        Z[:], Ga[:], Gb[:], Gn[:] = p["Z"], p["Ga"], p["Gb"], p["Gnorm"]
        start_row = int(p["rows_done"])
        print(f"resuming from row {start_row}")

    # only the perturbed layers need weight grads; everything else just passes activation grads through
    req = {n: p.requires_grad for n, p in model.named_parameters()}
    for p in model.parameters(): p.requires_grad_(False)
    for n in names: layers[n].weight.requires_grad_(True)

    try:
        for i in tqdm(range(start_row, grid_n), initial=start_row, total=grid_n):
            b = lin[i]
            for j, a in enumerate(lin):
                with torch.no_grad():
                    for n in names:
                        layers[n].weight.data.copy_(orig[n]).add_(delta[n], alpha=a).add_(eta[n], alpha=b)
                model.zero_grad(set_to_none=True)
                loss = crit(model(x), y)
                loss.backward()
                ga = gb = gn2 = 0.0
                with torch.no_grad():
                    for n in names:
                        g = layers[n].weight.grad
                        ga  += (g * delta[n]).sum().item()
                        gb  += (g * eta[n]).sum().item()
                        gn2 += (g * g).sum().item()
                Z[i, j], Ga[i, j], Gb[i, j], Gn[i, j] = loss.item(), ga, gb, gn2 ** 0.5
            if partial_path is not None and ((i + 1) % checkpoint_every == 0 or i == grid_n - 1):
                np.savez(partial_path, Z=Z, Ga=Ga, Gb=Gb, Gnorm=Gn, rows_done=i + 1)
    finally:
        with torch.no_grad():
            for n in names:
                layers[n].weight.data.copy_(orig[n])
        model.zero_grad(set_to_none=True)
        for n, p in model.named_parameters(): p.requires_grad_(req[n])
    return lin, Z, Ga, Gb, Gn

In [ ]:
t = TARGET
model_name, layer_idxs = LANDSCAPES[t["cfg"]]

if SKIP_EXISTING and paths["npz"].exists():
    d = np.load(paths["npz"])
    lin, Z, Ga, Gb, Gn = d["alphas"], d["Z"], d["Ga"], d["Gb"], d["Gnorm"]
    print("loaded existing", paths["npz"])
else:
    model = load_model(model_name)
    x, y = load_image(t["img_idx"])
    with torch.no_grad():
        loss0 = crit(model(x), y).item()
    delta, eta = make_directions(model, layer_idxs, t["dir_seed"])
    layer_names = [weighted_layers(model)[i][0] for i in layer_idxs]
    print(model_name, layer_names, f"L(0,0) = {loss0:.3f}")

    t0 = time.time()
    lin, Z, Ga, Gb, Gn = grad_surface(model, delta, eta, x, y, t["grid"], t["extent"],
                                      partial_path=paths["partial"], checkpoint_every=CHECKPOINT_EVERY)
    elapsed = time.time() - t0

    np.savez(paths["npz"], alphas=lin, betas=lin, Z=Z, Ga=Ga, Gb=Gb, Gnorm=Gn)
    wnid = val_ds.classes[val_ds.targets[t["img_idx"]]]
    M = np.hypot(Ga, Gb)
    meta = dict(**t, model=model_name, layer_idxs=layer_idxs, layer_names=layer_names,
                wnid=wnid, loss_origin=float(loss0),
                loss_min=float(Z.min()), loss_max=float(Z.max()),
                plane_grad_median=float(np.median(M)), plane_grad_max=float(M.max()),
                seconds=round(elapsed, 1),
                layout="Z/Ga/Gb/Gnorm[i,j] at alpha=alphas[j], beta=betas[i]; Ga=dL/dalpha, Gb=dL/dbeta (raw, ascent); "
                       "Gnorm=||full weight grad over perturbed layers||; texture orientation = imshow(rot90(Z.T))")
    paths["meta"].write_text(json.dumps(meta, indent=2))
    paths["partial"].unlink(missing_ok=True)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"done in {elapsed/60:.1f} min -> {paths['npz']}")

### Sanity checks

In [ ]:
# 1) Z should match the existing P33 hi-res render
p33_npy = P33_HIRES / t["cfg"] / f'{t["cfg"]}_img{t["img_idx"]:06d}_dir{t["dir_seed"]:04d}_grid{t["grid"]}_ext{t["extent"]:g}.npy'
if p33_npy.exists():
    Z33 = np.load(p33_npy)
    print(f"max |Z - Z_p33| = {np.abs(Z - Z33).max():.3e}   (Z range {Z.min():.2f} .. {Z.max():.1f})")
else:
    print("no P33 render found at", p33_npy)

# 2) autograd in-plane gradient vs finite differences of Z (checks the alpha/beta axis convention)
dZa = np.gradient(Z, lin, axis=1)   # d/dalpha
dZb = np.gradient(Z, lin, axis=0)   # d/dbeta
print(f"corr(Ga, dZ/dalpha_fd) = {np.corrcoef(Ga.ravel(), dZa.ravel())[0,1]:.5f}")
print(f"corr(Gb, dZ/dbeta_fd)  = {np.corrcoef(Gb.ravel(), dZb.ravel())[0,1]:.5f}")

# 3) how much of the gradient lives in the plane
M = np.hypot(Ga, Gb)
print(f"|grad_plane| median {np.median(M):.3f}  max {M.max():.2f}     |grad_full| median {np.median(Gn):.3f}  max {Gn.max():.2f}")
print(f"median in-plane fraction {np.median(M / (Gn + 1e-12)):.4f}")

### Preview (contour + descent quiver, every `STRIDE`-th point)

In [ ]:
STRIDE = 16   # 512 / 16 = 32 arrows per side

A, B = np.meshgrid(lin, lin)
Zc = np.clip(Z, None, t["loss_cap"])
M = np.hypot(Ga, Gb)
U, V = -Ga / (M + 1e-12), -Gb / (M + 1e-12)      # unit-length descent direction
s = STRIDE

fig = Figure(figsize=(8, 7)); FigureCanvasAgg(fig)
ax = fig.add_subplot(111)
ax.contourf(A, B, Zc, levels=25, cmap="viridis", alpha=0.85)
q = ax.quiver(A[::s, ::s], B[::s, ::s], U[::s, ::s], V[::s, ::s], np.log10(M[::s, ::s] + 1e-12),
              cmap="autumn", scale=45, width=0.003, pivot="mid")
fig.colorbar(q, ax=ax, shrink=0.85, label=r"$\log_{10}\,\|\nabla_{plane} L\|$")
ax.plot(0, 0, "w+", ms=12, mew=2)
ax.set_xlabel(r"$\alpha$"); ax.set_ylabel(r"$\beta$"); ax.set_aspect("equal")
ax.set_title(f'{t["cfg"]}   img {t["img_idx"]}   dirseed {t["dir_seed"]}   grid {t["grid"]}   ext {t["extent"]}\n'
             f'|g| med {np.median(M):.2f}   max {M.max():.1f}', fontsize=10)
fig.tight_layout()
fig.savefig(paths["preview"], dpi=150)
display(Image.open(paths["preview"]))

In [ ]:
# load-back snippet for the manim side
# d = np.load(".../p35_gradient_field/plain74_first4/plain74_first4_img020907_dir0037_grid512_ext1.5_gradfield.npz")
# alphas, betas, Z, Ga, Gb = d["alphas"], d["betas"], d["Z"], d["Ga"], d["Gb"]
# arrow at (alphas[j], betas[i], Z[i, j]) pointing along (-Ga[i, j], -Gb[i, j]) for descent